# Text To SQL Query Helper Tool using Hugging Face

Converts natural language text into SQL queries using a local HuggingFace model (TinyLlama) via LangChain.

In [ ]:
# Install required packages
!pip install -q transformers accelerate torch langchain-community langchain-core

## Load Hugging Face Model

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Load the TinyLlama text-generation pipeline
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    do_sample=False,        # greedy decoding - deterministic output
    temperature=1.0,        # ignored when do_sample=False, but required by some versions
    return_full_text=False  # return only newly generated tokens, not the prompt
)

# Pass the pipeline object directly
llm = HuggingFacePipeline(pipeline=pipe)

# Sanity check
print(f"Type of pipe        : {type(pipe)}")
print(f"Type of llm.pipeline: {type(llm.pipeline)}")
print(f"Same object?        : {llm.pipeline is pipe}")  # should print True

## Create Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """Create a SQL query snippet using the below text:
```{text}```
Just SQL query:"""

prompt = PromptTemplate(template=template, input_variables=["text"])

In [ ]:
# Build the LangChain pipeline: prompt -> llm
llm_chain = prompt | llm

## Run a Query

Change `text` below to any natural language description of a SQL operation.

In [ ]:
text = 'Extract all the unique values from column "age"'

result = llm_chain.invoke(text)

# invoke() returns a plain string via HuggingFacePipeline
print(result)

## More Example Queries

In [ ]:
examples = [
    'Get the top 5 customers by total purchase amount from the orders table',
    'Find all employees who joined after January 1 2020',
    'Count the number of products in each category',
]

for example in examples:
    print(f"Input : {example}")
    print(f"SQL   : {llm_chain.invoke(example).strip()}")
    print("-" * 60)